# 03 - Recurrencia municipal y tasa agrupada (analisis complementario)
Objetivo: distinguir municipios con senal persistente (aparecen en varios
anios con tasas altas) de fluctuacion aleatoria de un solo anio.

Insumo: data/processed/tasas_suicidio_municipal_chihuahua_2019_2024.csv
(de 02_population_merge.ipynb, con IC 95% Poisson por anio).

Nota metodologica: este notebook NO reemplaza el detalle anual con IC
(decision ya documentada en methodology.md) -- es un analisis complementario
para identificar candidatos a senal real, que luego se interpretan junto
con el detalle anual, no en vez de el.

In [ ]:
import pandas as pd
from scipy.stats import chi2

df_tasas = pd.read_csv('../data/processed/tasas_suicidio_municipal_chihuahua_2019_2024.csv', encoding='utf-8')
print(f'Filas cargadas (municipio x anio): {len(df_tasas):,}')
print(f'Municipios unicos: {df_tasas["NOM_MUN"].nunique()}')


## 1. Cuantos anios aparece cada municipio en el top 15 por tasa
Un municipio que aparece en 3+ de los 6 anios en el top 15 es mucho mas
dificil de explicar por azar que uno que aparece 1 sola vez.

In [ ]:
TOP_N = 15

df_tasas['en_top15_ese_anio'] = (
    df_tasas.groupby('anio')['tasa_por_100k']
    .rank(ascending=False, method='min') <= TOP_N
)

recurrencia = (
    df_tasas.groupby('NOM_MUN')['en_top15_ese_anio']
    .sum()
    .reset_index(name='anios_en_top15')
    .sort_values('anios_en_top15', ascending=False)
)
recurrencia[recurrencia['anios_en_top15'] > 0]


## 2. Tasa agrupada (pooled) 2019-2024 por municipio
Suma todos los casos y toda la poblacion-anio (persona-tiempo aproximado) de
los 6 anios, y calcula UNA tasa estable con su propio IC 95%. Esto reduce el
ruido de anio a anio, a costa de perder el detalle temporal (por eso es
COMPLEMENTARIO al detalle anual, no un reemplazo).

In [ ]:
def ic_poisson_95(casos):
    if casos == 0:
        lower = 0.0
    else:
        lower = chi2.ppf(0.025, 2 * casos) / 2
    upper = chi2.ppf(0.975, 2 * (casos + 1)) / 2
    return lower, upper

pooled = df_tasas.groupby('NOM_MUN').agg(
    casos_totales=('casos', 'sum'),
    poblacion_persona_anio=('poblacion_total', 'sum'),
).reset_index()

pooled['tasa_pooled_100k'] = round(
    pooled['casos_totales'] / pooled['poblacion_persona_anio'] * 100000, 2
)
ic_pooled = pooled['casos_totales'].apply(ic_poisson_95)
pooled['tasa_pooled_ic_inf'] = round(
    ic_pooled.apply(lambda x: x[0]) / pooled['poblacion_persona_anio'] * 100000, 2
)
pooled['tasa_pooled_ic_sup'] = round(
    ic_pooled.apply(lambda x: x[1]) / pooled['poblacion_persona_anio'] * 100000, 2
)

pooled.sort_values('tasa_pooled_100k', ascending=False).head(15)


## 3. Tabla final: recurrencia + tasa agrupada, lado a lado
Municipios ordenados primero por recurrencia (evidencia de persistencia),
luego por tasa agrupada. Los candidatos mas solidos a senal real son los que
tienen AMBOS: alta recurrencia Y tasa agrupada alta con IC angosto.

In [ ]:
tabla_final = recurrencia.merge(pooled, on='NOM_MUN')
tabla_final = tabla_final.sort_values(
    ['anios_en_top15', 'tasa_pooled_100k'], ascending=[False, False]
)
tabla_final.head(20)


## 4. Guardar tabla de recurrencia

In [ ]:
tabla_final.to_csv('../data/processed/recurrencia_municipal_chihuahua.csv', index=False, encoding='utf-8')
print(f'Guardado: {len(tabla_final):,} municipios')


## 5. Hallazgos
_Documentar aqui: cuales municipios muestran evidencia solida de senal real
(alta recurrencia + tasa agrupada con IC angosto) vs. cuales del top anual
eran ruido de un solo anio. Estos primeros son los candidatos a discutir a
profundidad en el articulo._